# 01 - Fit the 10-trait PCA (functional composition axes)

**Purpose.** Fit the principal component analysis that defines the three functional-composition
axes (PC1-PC3) used throughout the paper. Ten traits (nine leaf traits and canopy height) are
log-transformed, standardised, spatially de-trended and reduced to three components. The first
three components explain 85.3 % of the variance (the number quoted in the Results); the cell after
the spatial PCA prints it.

**Inputs.** `../data/pca_model/pca_sample_points.csv` - 39,350 random points over the contiguous
United States with the 30 m trait-map values and the NLCD class (run `../data/organize_zenodo_files.py`
first).

**Outputs.** `./results/scaler_rp_10traits.pkl` and `./results/PCA_model_rp_sa_10traits.pkl`. These
re-create the copies shipped in `../data/pca_model/`, which notebooks 02-03 and script 04 load by default.

**Figure.** None of the paper panels is drawn here (Fig. 2a-c come from notebooks 02 and 03, which
load the pickled model); the last cell shows a quick PC1-PC2 check plot.

Run with the working directory set to this folder.

In [ ]:
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform

DATA_DIR = '../data/pca_model'
OUT_DIR = './results'
os.makedirs(OUT_DIR, exist_ok=True)

# the ten traits entering the PCA (column names after renaming in the next cell)
trait_list = ['Carbon', 'Cellulose', 'Chlorophyll a + b', 'EWT', 'Lignin', 'Nitrogen',
              'NSC', 'Phenolics', 'SLA', 'Canopy Height']

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'pca_sample_points.csv'))
df.rename(columns={'ChlorophyllsArea': 'Chlorophyll a + b', 'canopy_height': 'Canopy Height'}, inplace=True)
# canopy height is an integer raster value; +1 avoids log(0) for zero-height pixels
df['Canopy Height'] = df['Canopy Height'] + 1

In [ ]:
df = df.dropna()

In [ ]:
# natural vegetation only: drop NLCD 81 (pasture/hay) and 82 (cultivated crops)
df = df.query('nlcd_class != 81 and nlcd_class != 82')

In [ ]:
# random subsample of 10,000 points (fixed seed) used for the PCA and the trait-space figures
df = df.sample(n=10000, random_state=1)

In [ ]:
nlcd_dict = {0: 'Open Water', 11: 'Developed, Open Space', 12: 'Developed, Low Intensity',
             21: 'Developed, Medium Intensity',
             22: 'Developed, High Intensity', 23: 'Developed, Open Space with Buildings',
             24: 'Developed, Open Space with Roads',
             31: 'Barren Land (Rock/Sand/Clay)', 41: 'Deciduous Forest', 42: 'Evergreen Forest', 43: 'Mixed Forest',
             51: 'Dwarf Scrub', 52: 'Shrub/Scrub', 71: 'Grassland/Herbaceous', 72: 'Sedge/Herbaceous', 73: 'Lichens',
             74: 'Moss', 81: 'Pasture/Hay', 82: 'Cultivated Crops', 90: 'Woody Wetlands',
             95: 'Emergent Herbaceous Wetlands'}
df['nlcd'] = df['nlcd_class'].map(nlcd_dict)

# Spatially de-trended PCA

Regular PCA versus PCA on the residuals of a spatial-lag regression (removes spatial autocorrelation).

In [ ]:
def calculate_moran_i(data, weights):
    """Moran's I statistic of a 1-D variable under a spatial weight matrix."""
    mean = np.mean(data)
    n = len(data)
    numerator = np.sum(weights * (data[:, np.newaxis] - mean) * (data[np.newaxis, :] - mean))
    denominator = np.sum((data - mean) ** 2)
    return (n / np.sum(weights)) * (numerator / denominator)


def remove_spatial_autocorrelation(data, coordinates):
    """Remove spatial autocorrelation from every column of `data`.

    For each column a spatial-lag regression (y ~ 1 + Wy, W = inverse-distance weights)
    is fitted and the residuals are returned.
    """
    # spatial weight matrix: inverse distance; 1 on the diagonal avoids division by zero
    distances = pdist(coordinates)
    dist_matrix = squareform(distances)
    weights = 1 / (dist_matrix + np.eye(dist_matrix.shape[0]))

    corrected_data = np.zeros_like(data)
    for i in range(data.shape[1]):
        y = data[:, i]
        W = weights

        # spatial lag of y
        spatial_lag = W.dot(y) / W.sum(axis=1)

        # spatial regression
        X = np.column_stack((np.ones_like(y), spatial_lag))
        beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)

        # residuals = data with the spatial autocorrelation removed
        corrected_data[:, i] = y - X.dot(beta)

    return corrected_data


def spatial_pca(data, coordinates, n_components=10):
    """PCA of spatially de-trended data (see remove_spatial_autocorrelation)."""
    corrected_data = remove_spatial_autocorrelation(data, coordinates)

    pca = PCA(n_components=n_components)
    pca.fit(corrected_data)

    return pca, pca.explained_variance_ratio_


# Compare a regular PCA with the spatially de-trended PCA (all 10 components)
X = StandardScaler().fit_transform(np.log(df[trait_list]))

pca_regular = PCA(n_components=10)
pca_regular.fit(X)

pca10, explained_variance = spatial_pca(X, df[['longitude', 'latitude']])

print('Regular PCA, explained variance ratio:', np.round(pca_regular.explained_variance_ratio_, 4))
print('Spatial PCA, explained variance ratio:', np.round(explained_variance, 4))
print('Difference (regular - spatial):       ',
      np.round(pca_regular.explained_variance_ratio_ - explained_variance, 4))

In [ ]:
# Cumulative variance explained by the first three spatially de-trended PCs.
# This is the 85.3 % reported in the paper.
cum_var_3 = explained_variance[:3].sum()
print('PC1-PC3 explained variance ratio:', np.round(explained_variance[:3], 4))
print(f'Cumulative variance explained by PC1-PC3: {cum_var_3 * 100:.1f} %')

# Trait functional groups

Used to colour the trait labels in the biplots.

In [ ]:
# functional group of each trait (colours the trait labels in the biplots)
function = {'Canopy Height': 'Plant/Leaf structure', 'Carbon': 'Plant/Leaf structure',
            'EWT': 'Plant/Leaf structure',
            'Nitrogen': 'Light capture and growth', 'NSC': 'Light capture and growth',
            'Chlorophyll a + b': 'Light capture and growth', 'SLA': 'Light capture and growth',
            'Phenolics': 'Defense', 'Cellulose': 'Defense', 'Lignin': 'Defense'}

function_color = {'Plant/Leaf structure': '#A626A4', 'Light capture and growth': '#009E73',
                  'Defense': '#4053D3'}

# Fit the final 3-component model

In [ ]:
# Final model: log -> standardise -> spatially de-trended PCA with 3 components
X = np.log(df[trait_list])
scaler = StandardScaler()
scaler.fit(X)
X_scale = scaler.transform(X)

pca, explained_variance = spatial_pca(X_scale, df[['longitude', 'latitude']], n_components=3)

# Orient the axes. The sign of a principal component is arbitrary, and the convention
# scikit-learn picks changed in v1.5 (svd_flip), so the orientation is fixed explicitly to
# the published model: PC1 positive towards Carbon, PC2 positive towards Nitrogen,
# PC3 negative towards Phenolics (axes read as in Diaz et al. 2016). With scikit-learn 1.4,
# which fitted the published model, this equalled negating all three components.
axis_sign = {0: ('Carbon', 1), 1: ('Nitrogen', 1), 2: ('Phenolics', -1)}
for k, (trait, sign) in axis_sign.items():
    if np.sign(pca.components_[k, trait_list.index(trait)]) != sign:
        pca.components_[k] = -pca.components_[k]
X_scale_reduced = pca.transform(X_scale)

print('Explained variance ratio (3-component model):', np.round(pca.explained_variance_ratio_, 4))
print('Loadings:')
print(pd.DataFrame(pca.components_.T, index=trait_list, columns=['PC1', 'PC2', 'PC3']).round(3))

In [ ]:
plt.style.use('default')
sns.set_style('whitegrid')
plt.style.use('bmh')
sns.set_context('paper')
plt.rcParams['font.family'] = ['Helvetica', 'Arial', 'DejaVu Sans']
from matplotlib import cm
import matplotlib
import matplotlib.patheffects as pe

norm = matplotlib.colors.Normalize(vmin=0, vmax=6)
rgba = cm.gist_ncar([norm(0), norm(1), norm(2), norm(3), norm(4), norm(5), norm(6)])

# rgba to list of hex colours, one per NLCD class
color_list = []
for i in range(rgba.shape[0]):
    color_list.append(matplotlib.colors.rgb2hex(rgba[i, :3]))

color_list[2] = '#006E00'
color_list[6] = '#7F7F7F'
color_list.append('#FF0000')


def PCA_2D_plot(score, coeff, labels=None, hue=None):
    fig, ax = plt.subplots(dpi=300, figsize=(5.5, 4.5))
    xs = score.iloc[:, 0]
    ys = score.iloc[:, 1]
    n = coeff.shape[0]
    sns.scatterplot(x=xs, y=ys, data=score, ax=ax, s=1, hue=hue,
                    **{'edgecolor': 'none', 'alpha': 0.8}, palette=color_list,
                    hue_order=['Deciduous Forest', 'Mixed Forest', 'Evergreen Forest', 'Shrub/Scrub',
                               'Grassland/Herbaceous', 'Woody Wetlands'
                               ])

    arrow_color = '#05445E'
    enlarge = 8
    for i in range(n):
        ax.annotate(labels[i], xy=(0, 0), xytext=(coeff[i, 0] * enlarge, coeff[i, 1] * enlarge),
                    color=function_color[function[labels[i]]],
                    arrowprops=dict(arrowstyle="<-", lw=1., color=arrow_color, linestyle='--')
                    , va='center', ha='center'
                    , path_effects=[pe.withStroke(linewidth=0.8, foreground="0.8")]
                    )

    ax.legend(loc='upper left', bbox_to_anchor=(0, 1), ncol=1, markerscale=3, frameon=False, fontsize='small')
    ax.grid(False)

    plt.xlabel("PC{} ({}%)".format(1, round(pca.explained_variance_ratio_[0] * 100, 1)))
    plt.ylabel("PC{} ({}%)".format(2, round(pca.explained_variance_ratio_[1] * 100, 1)))
    ax.set_facecolor('0.98')
    for spine in ax.spines.values():
        spine.set_edgecolor('k')
        spine.set_linewidth(1.5)
    return fig, ax


# quick check plot: PC1-PC2 biplot coloured by NLCD class (Fig. 2b itself is drawn in notebook 02)
X_scale_reduced_12 = pd.DataFrame(X_scale_reduced[:, :2], index=df.index, columns=['PC1', 'PC2'])
X_scale_reduced_12['nlcd'] = df['nlcd']
fig, ax = PCA_2D_plot(X_scale_reduced_12, np.transpose(pca.components_), labels=trait_list,
                      hue='nlcd')
ax.text(0.03, 0.13, 'Plant/Leaf structure', color=function_color['Plant/Leaf structure'], fontsize
='small', ha='left', va='top', transform=ax.transAxes, path_effects=[pe.withStroke(linewidth=0.8, foreground="0.8")])
ax.text(0.03, 0.09, 'Light capture and growth', color=function_color['Light capture and growth'], fontsize
='small', ha='left', va='top', transform=ax.transAxes, path_effects=[pe.withStroke(linewidth=0.8, foreground="0.8")])
ax.text(0.03, 0.05, 'Defense', color=function_color['Defense'], fontsize
='small', ha='left', va='top', transform=ax.transAxes, path_effects=[pe.withStroke(linewidth=0.8, foreground="0.8")])
fig.tight_layout()
plt.show()
plt.close(fig)

## Save the fitted scaler and PCA

In [ ]:
with open(os.path.join(OUT_DIR, 'scaler_rp_10traits.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
with open(os.path.join(OUT_DIR, 'PCA_model_rp_sa_10traits.pkl'), 'wb') as f:
    pickle.dump(pca, f)
print('saved scaler and PCA model to', OUT_DIR)